# PHA - Domain Expert Agent Demo

This notebook demonstrates the **Domain Expert Agent** from PHA.

The Domain Expert Agent uses the **ReAct framework** (via onetwo) for:
- Iterative reasoning about health questions
- Tool use (web search, data commons, code execution, reference ranges)
- Evidence-based health interpretation

## Requirements

For full ReAct functionality, install onetwo:
```bash
pip install git+https://github.com/google-deepmind/onetwo
```

The agent will still work without onetwo, but in a simplified mode.

## Setup

In [ ]:
# Install dependencies (uncomment as needed)
# !pip install git+https://github.com/google-deepmind/onetwo  # For ReAct support
# !pip install google-genai openai # LLM backends
# !pip install tavily-python  # Better search (optional)
# !pip install duckduckgo-search  # Free search fallback

In [ ]:
import os
import sys

# --- Configuration ---
# Set your API key and provider here
API_KEY = "your-api-key-here"
# Provider options: "gemini", "openai", "anthropic"
PROVIDER = "gemini"

# Optional: Tavily API key for Domain Expert Agent (search)
TAVILY_API_KEY = "your-tavily-api-key-here"


In [ ]:
# Check if onetwo is available
from pha.agents import is_react_available

print(f"OneTwo ReAct available: {is_react_available()}")
if not is_react_available():
    print("Note: Running in fallback mode without full ReAct support.")
    print("For full functionality, install: pip install git+https://github.com/google-deepmind/onetwo")

## Initialize the Domain Expert Agent

In [ ]:
from pha.agents import DomainExpertAgent

# Create the agent
# Default uses Tavily for high-quality search (requires TAVILY_API_KEY)
# Set search_backend='duckduckgo' for free but lower quality search
agent = DomainExpertAgent(
    search_backend='tavily',  # or 'duckduckgo' if you don't have a Tavily key
    tavily_api_key=TAVILY_API_KEY,
)

# Configure setup (uses global LLM config)
import glob

# Find exemplar notebooks (PHIA pattern)
exemplar_files = glob.glob('../few_shots/*.ipynb')
print(f"Found {len(exemplar_files)} exemplar notebooks")

agent.get_agent(
    api_key=API_KEY,
    provider=PROVIDER,
    exemplar_files=exemplar_files,
)

print("Domain Expert Agent initialized!")
print(f"Using ReAct: {is_react_available()}")


## Set User Health Data (Optional)

The agent can use user health data to provide personalized responses.

In [ ]:
# Set user health data manually
agent.set_user_health_data("""
## User Profile
- Age: 45
- Sex: Male
- Height: 178 cm
- Weight: 82 kg
- BMI: 25.9

## Recent Lab Results
- LDL Cholesterol: 142 mg/dL
- HDL Cholesterol: 52 mg/dL
- Total Cholesterol: 210 mg/dL
- Triglycerides: 145 mg/dL
- Fasting Glucose: 98 mg/dL
- HbA1c: 5.7%

## Wearable Data (7-day average)
- Daily Steps: 7,500
- Resting Heart Rate: 68 bpm
- Heart Rate Variability: 42 ms
- Sleep Duration: 6.5 hours
""")

print("User health data set!")

## Ask Health Questions

The agent will use ReAct reasoning to:
1. Think about the question
2. Use tools (search, calculations, reference ranges)
3. Observe results
4. Iterate until reaching an answer

In [ ]:
# Example 1: Interpret lab results
question = "Based on my lab results, should I be concerned about my cholesterol levels?"

print(f"Question: {question}")
print("\nAgent reasoning...")

response = agent.call_agent(question)
print(f"\nResponse:\n{response}")

In [ ]:
# Example 2: Ask about a specific metric
question = "What does my HbA1c of 5.7% indicate about my diabetes risk?"

print(f"Question: {question}")
print("\nAgent reasoning...")

response = agent.call_agent(question)
print(f"\nResponse:\n{response}")

In [ ]:
# Example 3: Ask about wearable metrics
question = "How does my resting heart rate and HRV compare to healthy norms for my age?"

print(f"Question: {question}")
print("\nAgent reasoning...")

response = agent.call_agent(question)
print(f"\nResponse:\n{response}")

In [ ]:
# Example 4: Comprehensive health assessment
question = """Given all my health data, what are the top 3 things I should focus on 
to improve my overall health? Please provide specific, actionable recommendations."""

print(f"Question: {question}")
print("\nAgent reasoning...")

response = agent.call_agent(question)
print(f"\nResponse:\n{response}")

## Load Health Data from Sample Files

In [ ]:
# Load sample data
from config import Settings
from pha.utils import load_persona

settings = Settings(data_dir='../data/sample')
summary_df, activities_df, profile_df, population_df = load_persona(settings=settings)

# Format recent data for the agent
recent_7_days = summary_df.tail(7)

health_summary = f"""
## User Profile
- Age: {profile_df['age'].iloc[0]}
- Sex: {profile_df['sex'].iloc[0]}
- Height: {profile_df['height_cm'].iloc[0]} cm
- Weight: {profile_df['weight_kg'].iloc[0]} kg

## 7-Day Wearable Summary (averages)
- Daily Steps: {recent_7_days['steps'].mean():.0f}
- Sleep Duration: {recent_7_days['sleep_minutes'].mean():.0f} minutes
- Resting Heart Rate: {recent_7_days['resting_heart_rate'].mean():.1f} bpm
- Heart Rate Variability: {recent_7_days['heart_rate_variability'].mean():.1f} ms
- Sleep Score: {recent_7_days['sleep_score'].mean():.1f}
"""

agent.set_user_health_data(health_summary)
print("Loaded sample health data:")
print(health_summary)

In [ ]:
# Ask a question about the loaded data
question = "Based on my wearable data, how is my sleep quality and what can I do to improve it?"

print(f"Question: {question}")
print("\nAgent reasoning...")

response = agent.call_agent(question)
print(f"\nResponse:\n{response}")

## Try Your Own Questions!

In [ ]:
# Ask your own health question
your_question = "What lifestyle changes would have the biggest impact on my cardiovascular health?"

response = agent.call_agent(your_question)
print(response)